In [1]:
print('hi')

hi


In [2]:
from pydantic import BaseModel, Field
from typing import Annotated, Sequence, TypedDict, Literal
from langchain_core.messages import BaseMessage, AIMessage
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import operator
from typing import List, TypedDict, Annotated, Sequence
from langgraph.graph.message import add_messages
from IPython.display import Image, display
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_community.tools import DuckDuckGoSearchRun
from langchain.tools import tool
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.tools import create_retriever_tool
from langsmith.client import Client

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
load_dotenv()
os.environ['OPENAI_API_KEY']=os.getenv('OPENAI_API_KEY')

In [4]:
llm=ChatOpenAI()
llm.invoke("What is the weather in Tokyo?")

AIMessage(content='The weather in Tokyo is currently partly cloudy with a temperature of 61°F (16°C).', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 14, 'total_tokens': 33, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DX48QHLLIWte3XQrbyvPBz6HWCIKu', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019dafed-c6c6-75d1-b743-44fff4b544d4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 19, 'total_tokens': 33, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [8]:
from langgraph.types import Command
from langgraph.prebuilt import create_react_agent ## In built class for react agent -> pass model name and tool


In [15]:
def add_number(state):
    result=state['num1']+state['num2']
    print(f"The sum of {state['num1']} and {state['num2']} is {result}")
    return Command(goto="multiply", update={"sum":result})

    # The command has some parameters out of which the goto and update are important.
    # goto means -> which function next we have to go to.
    # update means -> update the state with the new key value pair.

In [11]:
state={"num1":10, "num2":20}

In [16]:
add_number(state)

The sum of 10 and 20 is 30


Command(update={'sum': 30}, goto='multiply')

#### ### Creating dumy multi agent for Network / Supervisor Multi Agent

In [ ]:
@tool
def transfer_to_multiplication_expert():
    """Ask multiplication agent for help"""
    return

In [18]:
@tool
def transfer_to_addition_expert():
    """Ask addition agent for help"""
    return

In [19]:
llm_with_tool=llm.bind_tools([transfer_to_multiplication_expert])

In [20]:
response=llm_with_tool.invoke("hi")

In [21]:
response.content

'Hello! How can I assist you today?'

In [22]:
response.tool_calls

[]

In [ ]:
response=llm_with_tool.invoke("What is 2 multiply by 2?")

In [ ]:
response.content

''

In [26]:
response.tool_calls

[{'name': 'transfer_to_multiplication_expert',
  'args': {},
  'id': 'call_q8jbLa8hcp94AZGXCAjzERYL',
  'type': 'tool_call'}]

In [35]:
## AGENT 1

def addition_expert(state: MessagesState)-> Command[Literal["multiplication_expert", "__end__"]]:
    system_prompt=(
        "You are an addition expert. You can ask the multiplication expert for help with multiplication."
        "Always do your portion of calculation before the handing over the task to the multiplication expert."
    )

    messages=[{"role":"system", "content":system_prompt}] + state['messages']
    
    llm_with_tool=llm.bind_tools([transfer_to_multiplication_expert])
    ai_msg=llm_with_tool.invoke(messages)

    tool_call_id=ai_msg.tool_calls[-1]['id']
    
    if len[ai_msg.tool_calls]>0:
        tool_msg={
            "role": "tool",
            "content": "Successfully transferred",
            "tool_call_id": tool_call_id
        }        
    return {"messages": messages + [ai_msg, tool_msg]}

In [ ]:
## AGENT 2

def multiplication_expert(state: MessagesState)-> Command[Literal["addition_expert", "__end__"]]:
    pass

In [30]:
graph=StateGraph(MessagesState)

In [31]:
graph.add_node('addition_agent', addition_expert)

In [32]:
graph.add_node('multiplication_agent', multiplication_expert)

In [33]:
graph.add_edge(START, 'addition_agent')

In [ ]:
app=graph.compile()